# Breast Cancer Classification Pipeline

This notebook trains five classifiers on the scikit-learn Breast Cancer Wisconsin dataset, evaluates six classification metrics, exports a held-out test set, and saves each trained model in `model/`.

In [1]:
from pathlib import Path
import joblib
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
# Load the built-in Breast Cancer Wisconsin dataset.
base_dir = Path.cwd()
def load_data():
    data = load_breast_cancer(as_frame=True)
    X = data.data
    y = data.target  # 0 = malignant, 1 = benign
    return X, y, data.target_names


X, y, target_names = load_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, shuffle=True, random_state=42
)

test_data = X_test.copy()
test_data['target'] = y_test
test_data.to_csv(base_dir / 'test_data.csv', index=False)

print(f'Dataset shape: {X.shape}')
print(f'Target names: {list(target_names)}')
print(f'Training rows: {len(X_train)} | Test rows: {len(X_test)}')
print('Test data exported to test_data.csv')

Dataset shape: (569, 30)
Target names: [np.str_('malignant'), np.str_('benign')]
Training rows: 455 | Test rows: 114
Test data exported to test_data.csv


In [17]:
models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    'decision_tree': DecisionTreeClassifier(random_state=42),
    'knn': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', KNeighborsClassifier(n_neighbors=5)),
    ]),
    'naive_bayes': GaussianNB(),
    'random_forest': RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
}

model_dir = base_dir / 'model'
model_dir.mkdir(exist_ok=True)

results = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, predictions),
        'AUC': roc_auc_score(y_test, probabilities),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1 Score': f1_score(y_test, predictions, zero_division=0),
        'MCC': matthews_corrcoef(y_test, predictions),
    })

    joblib.dump(model, model_dir / f'{model_name}.joblib')

metrics = pd.DataFrame(results).set_index('Model').round(4)
metrics

,Accuracy,AUC,Precision,Recall,F1 Score,MCC
Model,,,,,,
logistic_regression,0.9825,0.9954,0.9861,0.9861,0.9861,0.9623
decision_tree,0.9123,0.9157,0.9559,0.9028,0.9286,0.8174
knn,0.9561,0.9788,0.9589,0.9722,0.9655,0.9054
naive_bayes,0.9386,0.9878,0.9452,0.9583,0.9517,0.8676
random_forest,0.9474,0.9937,0.9583,0.9583,0.9583,0.8869


## Saved Assets

- `test_data.csv`: the stratified held-out test partition, including IDs and diagnosis labels.
- `model/*.joblib`: one serialized fitted model per algorithm.

In [18]:
print('Saved model files:')
for path in sorted(model_dir.glob('*.joblib')):
    print(f'  {path.relative_to(base_dir)}')
print(f'  test_data.csv ({len(test_data)} rows)')

Saved model files:
  model/decision_tree.joblib
  model/knn.joblib
  model/logistic_regression.joblib
  model/naive_bayes.joblib
  model/random_forest.joblib
  test_data.csv (114 rows)
